# 03  Hypothesis Testing

Run the formal statistical tests to evaluate whether observed differences are real.

In [1]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from src.data_loader import load_cleaned, split_groups
from src.stats_utils import check_srm, two_prop_ztest, two_sample_ttest, power_analysis
import config

df         = load_cleaned()
ctrl, exp  = split_groups(df)

## 1. Sample Ratio Mismatch (SRM) check

Before trusting any results, verify that users were split correctly.

In [2]:
chi2, p_srm = check_srm(len(ctrl), len(exp))
print("SRM Check")
print(f"  Control   : {len(ctrl):,} users")
print(f"  Experiment: {len(exp):,} users")
print(f"  Chi2      : {chi2:.4f}")
print(f"  p-value   : {p_srm:.4f}")
print(f"  Result    : {'No SRM detected.' if p_srm > config.ALPHA else 'WARNING: SRM detected!'}")

SRM Check
  Control   : 4,800 users
  Experiment: 5,200 users
  Chi2      : 0.0000
  p-value   : 1.0000
  Result    : No SRM detected.


## 2. Primary metric- Conversion Rate (Z-test)

**H0:** p_experiment = p_control  
**H1:** p_experiment ≠ p_control  
α = 0.05

In [3]:
res = two_prop_ztest(
    ctrl['converted'].sum(), len(ctrl),
    exp['converted'].sum(),  len(exp)
)

print("Two-Proportion Z-Test: Conversion Rate")
print(f"  Control rate  : {res['p_ctrl']*100:.2f}%")
print(f"  Experiment rate: {res['p_exp']*100:.2f}%")
print(f"  Relative lift : +{res['lift_pct']:.1f}%")
print(f"  Z-statistic   : {res['z']:.4f}")
print(f"  P-value       : {res['p_value']:.6f}")
print(f"  95% CI (diff) : ({res['ci'][0]*100:.3f}%, {res['ci'][1]*100:.3f}%)")
print()
if res['p_value'] < config.ALPHA:
    print("Decision: REJECT H0 — result is statistically significant.")
else:
    print("Decision: Fail to reject H0.")

Two-Proportion Z-Test: Conversion Rate
  Control rate  : 11.27%
  Experiment rate: 14.33%
  Relative lift : +27.1%
  Z-statistic   : 4.5610
  P-value       : 0.000005
  95% CI (diff) : (1.743%, 4.369%)

Decision: REJECT H0 — result is statistically significant.


## 3. Secondary metrics - T-tests

In [4]:
for label, col in [("Revenue per User", "revenue"),
                   ("Session Time (min)", "session_time_min"),
                   ("Pages Viewed", "pages_viewed")]:
    r = two_sample_ttest(ctrl[col], exp[col])
    sig = "SIGNIFICANT" if r['p_value'] < config.ALPHA else "not significant"
    print(f"{label}")
    print(f"  Control mean   : {r['mean_ctrl']:.4f}")
    print(f"  Experiment mean: {r['mean_exp']:.4f}")
    print(f"  Change         : {r['pct_change']:+.1f}%")
    print(f"  P-value        : {r['p_value']:.5f}  ({sig})")
    print()

Revenue per User
  Control mean   : 5.6654
  Experiment mean: 8.6286
  Change         : +52.3%
  P-value        : 0.00000  (SIGNIFICANT)

Session Time (min)
  Control mean   : 8.8951
  Experiment mean: 10.5812
  Change         : +19.0%
  P-value        : 0.00000  (SIGNIFICANT)

Pages Viewed
  Control mean   : 4.0894
  Experiment mean: 4.6158
  Change         : +12.9%
  P-value        : 0.00000  (SIGNIFICANT)



## 4. Power analysis

In [5]:
req_n = power_analysis(config.BASELINE_CVR, config.MDE)
print("Power Analysis")
print(f"  Baseline CVR    : {config.BASELINE_CVR*100:.1f}%")
print(f"  Min detectable  : {config.MDE*100:.1f}%")
print(f"  Required n/group: {req_n:,}")
print(f"  Actual n/group  : {min(len(ctrl), len(exp)):,}")
print(f"  Adequately powered: {'Yes' if min(len(ctrl), len(exp)) >= req_n else 'No'}")

Power Analysis
  Baseline CVR    : 11.8%
  Min detectable  : 2.0%
  Required n/group: 4,172
  Actual n/group  : 4,800
  Adequately powered: Yes
